In [0]:
dbutils.widgets.removeAll()

In [0]:
%sql
SELECT COUNT(*) AS total_registros
FROM catalog_au.bronze.sfc_indicadores_raw;

In [0]:
# Leer conceptos transformados desde Silver

df_conceptos_transform = spark.table(
    "catalog_au.silver.sfc_conceptos_transform"
)

print(
    "Conceptos recibidos desde Silver:",
    df_conceptos_transform.count()
)

In [0]:
from pyspark.sql import functions as F

df_conceptos_load = df_conceptos_transform.select(
    "codigo_concepto",
    "nombre_concepto",
    "codigo_padre",
    "nivel_jerarquia",
    "ruta_jerarquia",
    "categoria_financiera",
    "unidad_medida",
    "es_concepto_hoja",
    "estado_concepto"
)

df_conceptos_load.createOrReplaceTempView(
    "vw_conceptos_load"
)

print("Conceptos preparados:", df_conceptos_load.count())

In [0]:
from pyspark.sql import functions as F

total = df_conceptos_load.count()

sk_unicas = (
    df_conceptos_load
    .select("sk_concepto")
    .distinct()
    .count()
)

sk_nulas = (
    df_conceptos_load
    .filter(F.col("sk_concepto").isNull())
    .count()
)

codigos_unicos = (
    df_conceptos_load
    .select("codigo_concepto")
    .distinct()
    .count()
)

print("Total conceptos:", total)
print("SK únicas:", sk_unicas)
print("SK nulas:", sk_nulas)
print("Códigos únicos:", codigos_unicos)

assert total == 589
assert sk_unicas == total
assert sk_nulas == 0
assert codigos_unicos == total

print("VALIDACIÓN PRE-LOAD: OK")

In [0]:
tabla_gold = "catalog_au.golden.dim_concepto_financiero"

df_gold = spark.table(tabla_gold)

print("Registros actuales en Golden:", df_gold.count())

df_gold.printSchema()

display(
    spark.sql(
        f"SHOW CREATE TABLE {tabla_gold}"
    )
)

In [0]:
df_conceptos_load.createOrReplaceTempView(
    "vw_conceptos_load"
)

print("Vista temporal creada: vw_conceptos_load")

display(
    spark.sql("""
        SELECT
            sk_concepto,
            codigo_concepto,
            nombre_concepto,
            codigo_padre,
            nivel_jerarquia
        FROM vw_conceptos_load
        ORDER BY sk_concepto
        LIMIT 10
    """)
)

In [0]:
spark.sql("""
MERGE INTO catalog_au.golden.dim_concepto_financiero AS target

USING vw_conceptos_load AS source

ON target.codigo_concepto = source.codigo_concepto

WHEN MATCHED AND NOT (
    target.nombre_concepto <=> source.nombre_concepto
    AND target.codigo_padre <=> source.codigo_padre
    AND target.nivel_jerarquia <=> source.nivel_jerarquia
    AND target.ruta_jerarquia <=> source.ruta_jerarquia
    AND target.categoria_financiera <=> source.categoria_financiera
    AND target.unidad_medida <=> source.unidad_medida
    AND target.es_concepto_hoja <=> source.es_concepto_hoja
    AND target.estado_concepto <=> source.estado_concepto
)
THEN UPDATE SET
    target.nombre_concepto = source.nombre_concepto,
    target.codigo_padre = source.codigo_padre,
    target.nivel_jerarquia = source.nivel_jerarquia,
    target.ruta_jerarquia = source.ruta_jerarquia,
    target.categoria_financiera = source.categoria_financiera,
    target.unidad_medida = source.unidad_medida,
    target.es_concepto_hoja = source.es_concepto_hoja,
    target.estado_concepto = source.estado_concepto,
    target.fecha_actualizacion = current_timestamp()

WHEN NOT MATCHED THEN INSERT (
    codigo_concepto,
    nombre_concepto,
    codigo_padre,
    nivel_jerarquia,
    ruta_jerarquia,
    categoria_financiera,
    unidad_medida,
    es_concepto_hoja,
    estado_concepto,
    fecha_creacion,
    fecha_actualizacion
)
VALUES (
    source.codigo_concepto,
    source.nombre_concepto,
    source.codigo_padre,
    source.nivel_jerarquia,
    source.ruta_jerarquia,
    source.categoria_financiera,
    source.unidad_medida,
    source.es_concepto_hoja,
    source.estado_concepto,
    current_timestamp(),
    current_timestamp()
)
""")

print("MERGE INTO Golden ejecutado correctamente")

In [0]:
%sql
SELECT
    COUNT(*) AS total_conceptos,
    COUNT(DISTINCT codigo_concepto) AS codigos_unicos,
    COUNT(DISTINCT sk_concepto) AS claves_unicas,
    SUM(CASE WHEN sk_concepto IS NULL THEN 1 ELSE 0 END)
        AS claves_nulas
FROM catalog_au.golden.dim_concepto_financiero;

In [0]:
df_antes = spark.sql("""
    SELECT
        COUNT(*) AS total,
        MAX(fecha_actualizacion) AS ultima_actualizacion
    FROM catalog_au.golden.dim_concepto_financiero
""")

display(df_antes)

In [0]:
df_despues = spark.sql("""
    SELECT
        COUNT(*) AS total,
        MAX(fecha_actualizacion) AS ultima_actualizacion
    FROM catalog_au.golden.dim_concepto_financiero
""")

display(df_despues)